# Motor ramp characterisation

The treadmill analysis discards the acceleration at the start of each trial with a
_model_ rather than a measurement (`cottage_analysis.analysis.treadmill.process_imaging_df`):

```python
acc_frames = ((acceleration_time * motor_speeds + 0.5) * frame_rate).astype(int)
```

with `acceleration_time = 0.13` s per cm/s and a fixed 0.5 s margin. Trial 56 of
`PZAG17.3a_S20250402` (30.5 cm/s) shows it failing: the running speed is still rising
~0.4 s after the analysed window opens, and `max_abs_rs2motor_diff_ratio` cannot flag
those frames because the top of a ramp is by construction within 30% of the plateau.

This notebook does two things, and only these two:

1. **measure the acceleration** of the belt, and establish whether it is a constant
   (sections A–C);
2. **detect the start of the plateau** per trial, on the signal the pipeline already has
   (section D).

Detecting the plateau directly replaces the indirect route of estimating when the belt
started moving and adding `v/a`. The onset latency is therefore not analysed here at all —
it is a nuisance parameter of the trapezoid fit in section B and nothing more.

Three data streams are used, all for the `SpheresTubeMotor` recordings:

| stream                                                      | rate    | gives                           |
| ----------------------------------------------------------- | ------- | ------------------------------- |
| `FrameLog.csv` via `get_frame_log` (`MotorSps`, `HarpTime`) | ~143 Hz | the motor command               |
| harp `analog_time` / `rotary_meter`                         | ~1 kHz  | ground-truth running speed      |
| `RS_stim` from `trials_df`                                  | 15.2 Hz | what the analysis pipeline sees |

Reading the frame log directly avoids the photodiode synchronisation, so a whole session
loads in seconds rather than minutes.


In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from scipy.signal import savgol_filter

import flexiznam as flz
from cottage_analysis.io_module import harp
from cottage_analysis.io_module.visstim import get_frame_log
from cottage_analysis.analysis import spheres, treadmill

cm = 1 / 2.54
PROJECT = "colasa_3d-vision_revisions"
EXAMPLE_SESSION = "PZAG17.3a_S20250402"
FS_VOL = 15.219  # imaging volume rate of the example session
MODEL_ACC_TIME, MODEL_MARGIN = 0.13, 0.5  # the current model in process_imaging_df
# Belt acceleration. Measured in section C, which asserts this value against the fits;
# the detector of section D needs it for its physical lower bound and its rise limit.
A_FIXED = 7.75
# Plateau detector (section D): the expected speed profile is slid over each trial and
# scored with a capped loss, so a sample that misses the template by 50% and one that
# misses by 100% cost the same -- a blocked wheel is "bad" but not catastrophically so.
CAP_REL, CAP_ABS = 0.10, 1.5   # cap = max(CAP_REL * v, CAP_ABS), in cm/s
SHIFT_STEP = 0.01              # s; a quarter of an imaging volume is plenty
MIN_PLATEAU_S = 1.0            # require this much plateau left after the detected start

flm = flz.get_flexilims_session(project_id=PROJECT)
save_folder = flz.get_processed_path(f"{PROJECT}/analysis/treadmill")
save_folder.mkdir(parents=True, exist_ok=True)

## Loading

`load_recording` returns the command edges (from the frame log) and the 1 kHz running
speed (from the raw rotary encoder), for one treadmill recording.


In [ ]:
def find_treadmill_recordings(session_name, flexilims_session):
    """Names of the SpheresTubeMotor recordings of a session."""
    exp = flz.get_entity(
        datatype="session", name=session_name, flexilims_session=flexilims_session
    )
    recs = flz.get_entities(
        datatype="recording",
        origin_id=exp["id"],
        query_key="recording_type",
        query_value="two_photon",
        flexilims_session=flexilims_session,
    )
    recs = recs[recs.name.str.contains("SpheresTubeMotor")]
    if "exclude_reason" in recs.columns:
        recs = recs[recs["exclude_reason"].isna()]
    return list(recs.name)


def load_recording(recording_name, flexilims_session, speed_window=0.1):
    """Load the motor command and the 1 kHz running speed of one treadmill recording.

    Args:
        recording_name (str): name of the SpheresTubeMotor recording.
        flexilims_session (flexilims_session): flexilims session.
        speed_window (float, optional): width in s of the Savitzky-Golay window used to
            differentiate the wheel position. Defaults to 0.1.

    Returns:
        dict: `time` and `rs` of the 1 kHz running speed (s, cm/s), `position` (cm), and
            `epochs`, a DataFrame with one row per motor step: command onset and offset
            times and the nominal plateau speed in cm/s.
    """
    recording, harp_recording, _ = spheres.get_relevant_recordings(
        recording_name, flexilims_session, harp_is_in_recording=True, use_onix=False
    )
    frame_log = get_frame_log(
        flexilims_session, harp_recording=harp_recording, vis_stim_recording=recording
    )
    assert "MotorSps" in frame_log.columns, f"No MotorSps in {recording_name} frame log"
    npz, _ = harp.load_harpmessage(
        recording_name, flexilims_session=flexilims_session, conflicts="skip"
    )

    # 1 kHz running speed: differentiate the cumulative wheel position
    time = np.asarray(npz["analog_time"])
    position = np.cumsum(np.asarray(npz["rotary_meter"])) * 100  # cm
    dt = np.median(np.diff(time))
    window = int(round(speed_window / dt)) // 2 * 2 + 1  # odd
    rs = savgol_filter(position, window, 2, deriv=1, delta=dt)  # cm/s

    # Command edges, from the frame log rather than the imaging volumes
    sps = frame_log.MotorSps.values.astype(float)
    frame_time = frame_log.HarpTime.values
    on = np.where(np.diff((sps > 0).astype(int)) == 1)[0] + 1
    off = np.where(np.diff((sps > 0).astype(int)) == -1)[0]
    n = min(len(on), len(off))
    on, off = on[:n], off[:n]
    targets = np.array([np.nanmedian(sps[o : e + 1]) for o, e in zip(on, off)])
    plateau = np.array(
        [
            treadmill.ACTUAL_MOTOR_SPEED.get(int(round(s)), np.nan)
            for s in np.round(treadmill.sps2speed(targets))
        ]
    )
    epochs = pd.DataFrame(
        dict(
            trial=np.arange(n),
            command_on=frame_time[on],
            command_off=frame_time[off],
            target_sps=targets,
            plateau_speed=plateau,
        )
    )
    return dict(
        recording=recording_name,
        time=time,
        rs=rs,
        position=position,
        frame_time=frame_time,
        sps=sps,
        epochs=epochs,
        frame_rate=1 / np.median(np.diff(frame_time)),
    )

In [ ]:
example_recordings = find_treadmill_recordings(EXAMPLE_SESSION, flm)
print(example_recordings)
rec = load_recording(example_recordings[0], flm)
print(
    f"frame log at {rec['frame_rate']:.0f} Hz, "
    f"{len(rec['time'])} analog samples at "
    f"{1/np.median(np.diff(rec['time'])):.0f} Hz"
)
print(
    f"{len(rec['epochs'])} motor steps, speeds "
    f"{np.unique(rec['epochs'].plateau_speed.dropna())}"
)

## A. Is the acceleration commanded?

If the controller ramps its own step rate, the acceleration is a deterministic property
of the command and there is nothing to measure. Checked at frame-log resolution (~7 ms),
which is where an `imaging_df`-based check would fail: `MotorSpeed` maps only the plateau
`sps` values through `ACTUAL_MOTOR_SPEED`, so intermediate values would silently become
NaN.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14 * cm, 5 * cm), sharey=False)
ep = rec["epochs"].dropna(subset=["plateau_speed"])
for speed, group in ep.groupby("plateau_speed"):
    trial = group.iloc[0]
    i_on = np.searchsorted(rec["frame_time"], trial.command_on)
    sl = slice(i_on - 3, i_on + 8)
    t = (rec["frame_time"][sl] - trial.command_on) * 1e3
    axes[0].step(t, rec["sps"][sl], where="post", label=f"{speed:.0f} cm/s")
    i_off = np.searchsorted(rec["frame_time"], trial.command_off)
    sl = slice(i_off - 3, i_off + 8)
    axes[1].step(
        (rec["frame_time"][sl] - trial.command_off) * 1e3,
        rec["sps"][sl],
        where="post",
    )
for ax, title in zip(axes, ["Command onset", "Command offset"]):
    ax.set_xlabel("Time from command edge (ms)")
    ax.set_title(title, fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("MotorSps")
axes[0].legend(fontsize=6, frameon=False)
plt.tight_layout()

# Count intermediate values between 0 and target
n_intermediate = []
for _, trial in ep.iterrows():
    i_on = np.searchsorted(rec["frame_time"], trial.command_on)
    w = rec["sps"][i_on : i_on + 20]
    k = np.argmax(w >= trial.target_sps)
    n_intermediate.append(k)
print(
    f"monitor frames between the first non-zero sps and the target: "
    f"{np.unique(n_intermediate)}"
)

**The command is a hard step.** `MotorSps` goes from 0 to its target in a single
frame-log sample (7 ms) with no intermediate values, at both edges and at every speed.
Whatever ramping we see in the running speed comes from the controller's internal
acceleration profile and/or the mouse, and the wheel encoder measures the two combined.


## B. Trapezoid fit, per trial, at 1 kHz

Five free parameters per trial: onset latency `t_on`, acceleration `a`, observed plateau
`v`, the time `t_off` at which the fall starts, and deceleration `d`. Only `a` is read out
— the other four are nuisance parameters the model needs in order to describe the trace.
A robust (`soft_l1`) loss keeps a mouse fighting the belt from dragging the whole fit.

Fit quality is scored as residual RMS **normalised by the plateau speed** — plain R² is
misleading here because most of the trace is a flat plateau, so at low speeds it is
dominated by noise on a small signal.


In [ ]:
def trapezoid(t, t_on, a, v, t_off, d):
    """Rise / plateau / fall, with t relative to the command onset."""
    return np.clip(np.minimum(a * (t - t_on), v - d * (t - t_off)), 0, v)


def fit_trapezoid(t, rs, plateau_speed, command_duration):
    """Fit the trapezoid to one trial. Returns a dict of parameters and fit quality."""
    p0 = [0.0, 1 / MODEL_ACC_TIME, plateau_speed, command_duration, 1 / MODEL_ACC_TIME]
    bounds = (
        [-1.0, 0.1, 0.2 * plateau_speed, command_duration - 3, 0.1],
        [5.0, 500.0, 3.0 * plateau_speed, command_duration + 3, 500.0],
    )
    res = least_squares(
        lambda p: trapezoid(t, *p) - rs,
        p0,
        bounds=bounds,
        loss="soft_l1",
        f_scale=max(0.05 * plateau_speed, 1.0),
        max_nfev=2000,
    )
    t_on, a, v, t_off, d = res.x
    resid = rs - trapezoid(t, *res.x)
    out = dict(
        t_on=t_on,
        a=a,
        v=v,
        t_off=t_off,
        d=d,
        ramp_duration=v / a,
        nrms=np.sqrt(np.mean(resid**2)) / plateau_speed,
    )
    # Standard error of `a`, from the Jacobian at the solution. Needed to tell genuine
    # trial-to-trial spread of the acceleration from estimation noise.
    try:
        dof = max(len(t) - len(res.x), 1)
        cov = np.linalg.inv(res.jac.T @ res.jac) * (2 * res.cost / dof)
        out["a_se"] = np.sqrt(abs(cov[1, 1]))
    except np.linalg.LinAlgError:
        out["a_se"] = np.nan
    return out


def fit_recording(rec, pre=1.0, post=2.0):
    """Fit the trapezoid to every trial of a recording, on the 1 kHz running speed."""
    rows = []
    for _, trial in rec["epochs"].iterrows():
        if not np.isfinite(trial.plateau_speed):
            continue
        t0, duration = trial.command_on, trial.command_off - trial.command_on
        sel = (rec["time"] >= t0 - pre) & (rec["time"] <= trial.command_off + post)
        if sel.sum() < 100:
            continue
        row = dict(
            recording=rec["recording"],
            trial=int(trial.trial),
            plateau_speed=trial.plateau_speed,
            command_duration=duration,
        )
        row.update(
            fit_trapezoid(
                rec["time"][sel] - t0, rec["rs"][sel], trial.plateau_speed, duration
            )
        )
        # Noise on the settled plateau. Not used to measure `a`, but it is what sets the
        # tolerance the detector of section D can afford.
        settled = rec["rs"][sel][
            (rec["time"][sel] - t0 > row["t_on"] + row["ramp_duration"] + 0.5)
            & (rec["time"][sel] - t0 < duration - 0.5)
        ]
        row["plateau_noise"] = settled.std() if len(settled) > 10 else np.nan
        rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
fits = fit_recording(rec)
GOOD_NRMS = 0.15
fits["good"] = fits.nrms < GOOD_NRMS
good = fits[fits.good]
print(f"{fits.good.sum()}/{len(fits)} trials with normalised residual < {GOOD_NRMS}")
print(
    fits.groupby("plateau_speed")
    .agg(n=("trial", "count"), frac_good=("good", "mean"), nrms_med=("nrms", "median"))
    .round(2)
    .to_string()
)

Pass rates are strongly speed-dependent (0.86 at 61 cm/s, 0.13 at 7.6 cm/s): at low
speed the mouse's own stepping noise is a large fraction of a small plateau. The
conclusions below therefore rest mainly on the 15.25–61 cm/s trials, and the low-speed
numbers are reported but should not be over-read.


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20 * cm, 4.5 * cm), sharex=False)
for ax, (speed, group) in zip(axes, fits.groupby("plateau_speed")):
    for _, trial in group.iterrows():
        ep = rec["epochs"].iloc[int(trial.trial)]
        sel = (rec["time"] >= ep.command_on - 1) & (
            rec["time"] <= ep.command_on + 2 + trial.ramp_duration
        )
        ax.plot(
            rec["time"][sel] - ep.command_on,
            rec["rs"][sel],
            color="grey" if trial.good else "lightcoral",
            lw=0.4,
            alpha=0.6,
        )
    med = group[group.good]
    if len(med):
        t = np.linspace(-1, 2 + med.ramp_duration.median(), 500)
        ax.plot(
            t,
            trapezoid(
                t,
                med.t_on.median(),
                med.a.median(),
                med.v.median(),
                1e6,
                med.d.median(),
            ),
            color="dodgerblue",
            lw=1.5,
        )
    ax.axvline(0, color="k", ls=":", lw=0.5)
    ax.axvline(MODEL_ACC_TIME * speed + MODEL_MARGIN, color="darkred", ls="--", lw=0.8)
    ax.set_title(f"{speed:.1f} cm/s", fontsize=8)
    ax.set_xlabel("Time from command (s)")
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("Running speed (cm/s)")
plt.tight_layout()
fig.savefig(save_folder / "ramp_overlay.pdf", bbox_inches="tight", transparent=True)

Blue is the median fitted trapezoid, dashed red the point at which the current
model opens the analysed window. Red traces are the trials that failed the fit-quality
cut.


## C. Is the acceleration a constant?

Two things have to hold for `a` to be usable as a constant by the detector: it must not
depend on the commanded speed, and it must not depend on the animal. The wheel encoder
measures belt **and** mouse, so the second test is what separates a controller property
from a behavioural one.


In [ ]:
constancy = good.groupby("plateau_speed").agg(
    n=("trial", "count"),
    a_med=("a", "median"),
    a_iqr=("a", lambda x: x.quantile(0.75) - x.quantile(0.25)),
    a_se_med=("a_se", "median"),
    s_per_cms=("a", lambda x: 1 / x.median()),
    ramp_med=("ramp_duration", "median"),
    plateau_noise=("plateau_noise", "median"),
)
print(constancy.round(3).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(8 * cm, 5 * cm))
speeds = np.array(sorted(good.plateau_speed.unique()))
for i, speed in enumerate(speeds):
    vals = good[good.plateau_speed == speed].a
    ax.plot(
        i + np.random.uniform(-0.15, 0.15, len(vals)),
        vals,
        ".",
        color="grey",
        ms=2,
        alpha=0.6,
    )
    ax.plot([i - 0.3, i + 0.3], [vals.median()] * 2, color="dodgerblue", lw=2)
ax.axhline(1 / MODEL_ACC_TIME, color="darkred", ls="--", lw=0.8)
ax.set_xticks(range(len(speeds)))
ax.set_xticklabels([f"{s:.0f}" for s in speeds], fontsize=6)
ax.set_xlabel("Motor speed (cm/s)")
ax.set_ylabel("Acceleration (cm/s$^2$)", fontsize=7)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
fig.savefig(
    save_folder / "ramp_acceleration.pdf", bbox_inches="tight", transparent=True
)

Dashed red is `1/0.13 = 7.7` cm/s², the constant implied by the current model. It sits
right on the measured medians — the model's acceleration constant is empirically correct.


In [ ]:
# Ramp duration vs speed: constant acceleration predicts a line through the origin
fig, ax = plt.subplots(figsize=(8 * cm, 7 * cm))
ax.plot(
    good.plateau_speed,
    good.ramp_duration,
    ".",
    color="grey",
    ms=3,
    alpha=0.5,
    label="Trials (v/a)",
)
v = np.linspace(0, 65, 100)
a_med = good[good.plateau_speed >= 15].a.median()
ax.plot(v, v / a_med, "k-", lw=1, label=f"Constant a = {a_med:.2f} cm/s$^2$")
ax.axhline(
    good[good.plateau_speed >= 15].ramp_duration.median(),
    color="green",
    ls=":",
    lw=1,
    label="Constant ramp duration",
)
ax.set_xlabel("Motor speed (cm/s)")
ax.set_ylabel("Ramp duration v/a (s)")
ax.legend(fontsize=5, frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
fig.savefig(
    save_folder / "ramp_duration_vs_speed.pdf", bbox_inches="tight", transparent=True
)

### Across mice


In [ ]:
ALL_SESSIONS = [
    "PZAG16.3b_S20250401",
    "PZAG16.3c_S20250401",
    "PZAG17.3a_S20250402",
    "PZAH17.1e_S20250403",
]

all_fits = []
for session_name in ALL_SESSIONS:
    for recording_name in find_treadmill_recordings(session_name, flm):
        rec_i = load_recording(recording_name, flm)
        f = fit_recording(rec_i)
        f["session"] = session_name
        f["mouse"] = session_name.split("_")[0]
        all_fits.append(f)
all_fits = pd.concat(all_fits, ignore_index=True)
all_fits["good"] = all_fits.nrms < GOOD_NRMS
print(
    f"{len(all_fits)} trials, {all_fits.good.sum()} good fits, "
    f"{all_fits.mouse.nunique()} mice"
)

In [ ]:
all_good = all_fits[all_fits.good]
print("acceleration a (cm/s2), median per mouse and speed")
print(
    all_good.pivot_table(
        index="mouse", columns="plateau_speed", values="a", aggfunc="median"
    )
    .round(2)
    .to_string()
)
print("\nn trials")
print(
    all_good.pivot_table(
        index="mouse", columns="plateau_speed", values="a", aggfunc="count"
    )
    .fillna(0)
    .astype(int)
    .to_string()
)

# Pool the speeds where the speed fit passes on a decent fraction of trials; below
# 15 cm/s the ramp is too short and too noisy to constrain the slope (see section B).
reliable = all_good[all_good.plateau_speed >= 15]
per_mouse = reliable.groupby("mouse").agg(
    n=("a", "count"),
    a_med=("a", "median"),
    a_iqr=("a", lambda x: x.quantile(0.75) - x.quantile(0.25)),
    s_per_cms=("a", lambda x: 1 / x.median()),
)
print("\npooled over speeds >= 15 cm/s")
print(per_mouse.round(3).to_string())
print(
    f"\nbetween-mouse spread of median a: {per_mouse.a_med.std():.3f} cm/s2 "
    f"({100*per_mouse.a_med.std()/reliable.a.median():.1f}% of the grand median)"
)

# The detector uses A_FIXED for its physical lower bound and its rise limit, so a wrong
# value silently moves every detection. Derive it here and fail loudly on disagreement
# rather than trusting the number typed into the setup cell.
A_MEASURED = round(reliable.a.median(), 2)
print(
    f"\nmeasured a = {A_MEASURED} cm/s2 "
    f"(1/a = {1/A_MEASURED:.4f} s per cm/s); setup cell has A_FIXED = {A_FIXED}"
)
assert abs(A_MEASURED - A_FIXED) < 0.5, (
    f"A_FIXED={A_FIXED} disagrees with the measured {A_MEASURED} cm/s2 - "
    "update the setup cell and re-run section D"
)

In [ ]:
fig, ax = plt.subplots(figsize=(8 * cm, 5 * cm))
mice = sorted(reliable.mouse.unique())
for i, mouse in enumerate(mice):
    vals = reliable[reliable.mouse == mouse].a
    ax.plot(
        i + np.random.uniform(-0.15, 0.15, len(vals)),
        vals,
        ".",
        color="grey",
        ms=2,
        alpha=0.5,
    )
    ax.plot([i - 0.3, i + 0.3], [vals.median()] * 2, color="dodgerblue", lw=2)
ax.axhline(1 / MODEL_ACC_TIME, color="darkred", ls="--", lw=0.8)
ax.set_xticks(range(len(mice)))
ax.set_xticklabels(mice, fontsize=6, rotation=30, ha="right")
ax.set_ylabel("Acceleration (cm/s$^2$)", fontsize=7)
ax.set_ylim(6, 10)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
fig.savefig(save_folder / "ramp_across_mice.pdf", bbox_inches="tight", transparent=True)

## D. Detecting the start of the plateau

What the pipeline needs is **when the mouse is following the target speed** — the point at
which the analysed window should open.

With `a` established as a constant and `v` commanded, the expected speed profile of a trial
has exactly **one** unknown: when the belt actually started. So rather than testing a local
criterion sample by sample, generate the expected time course

$$\text{template}(t; s) = \mathrm{clip}\big(a\,(t - s),\ 0,\ v\big)$$

slide it across the trial, and take the offset `s` that matches best. The plateau starts at
`s + v/a`.

Two properties come for free. Requiring `s >= 0` — the belt cannot move before the command —
makes `t_plateau >= v/a` **true by construction** rather than a bound imposed after the fact.
And the whole trace constrains the answer, not just the few samples around the transition.

**The loss has to saturate.** If the mouse blocks the wheel, those samples are bad, but it
should not matter whether they are 50% or 100% off — otherwise a single blocked episode
dominates the fit and drags `s` seconds away. So the per-sample cost is capped:

$$
\text{cost}(s) = \frac{1}{N}\sum_i \min\big(|rs_i - \text{template}(t_i; s)|,\ c\big) \Big/ c,
\qquad c = \max(0.10\,v,\ 1.5\ \text{cm/s})
$$

Dividing by `c` puts `cost` on [0, 1] and makes it comparable across speeds, so it doubles
as a per-trial goodness-of-match number. It is reported but **not** used as a criterion —
detection is `s + v/a` unconditionally.


In [ ]:
RECORDING_CACHE = {}


def get_recording(recording_name):
    """Load a recording once and keep it, so the plots below can mix recordings."""
    if recording_name not in RECORDING_CACHE:
        RECORDING_CACHE[recording_name] = load_recording(recording_name, flm)
    return RECORDING_CACHE[recording_name]


def ramp_template(t, shift, plateau_speed, acceleration=A_FIXED):
    """Expected running speed: rise at `acceleration` from `shift`, then hold the target."""
    return np.clip(acceleration * (t - shift), 0.0, plateau_speed)


def matchramp_template(
    t,
    rs,
    plateau_speed,
    duration,
    acceleration=A_FIXED,
    cap_rel=CAP_REL,
    cap_abs=CAP_ABS,
    shift_step=SHIFT_STEP,
    min_plateau=MIN_PLATEAU_S,
    pre=0.5,
    max_samples=3000,
):
    """Slide the expected speed profile over one trial and return the best match.

    Args:
        t (np.array): time relative to the motor command, in s.
        rs (np.array): running speed in cm/s.
        plateau_speed (float): commanded speed in cm/s.
        duration (float): command duration in s. Passed explicitly rather than taken from
            `t[-1]`, because `trial_signals` returns samples beyond the command end.
        acceleration (float, optional): the constant measured in section C.
        cap_rel, cap_abs (float, optional): the per-sample cost saturates at
            `max(cap_rel * plateau_speed, cap_abs)` cm/s. The absolute floor matters at
            3.8 cm/s, where the plateau noise alone is ~11% of the commanded speed and a
            purely relative cap would be exceeded by every sample.
        shift_step (float, optional): resolution of the search over onset times.
        min_plateau (float, optional): reject the trial if less than this much plateau
            would remain after the detected start.
        pre (float, optional): seconds before the command included in the match. The
            template is zero there, so it anchors the baseline.
        max_samples (int, optional): decimate longer traces. The loss is a mean, so this is
            unbiased -- and it is required, not an optimisation: the undecimated 1 kHz
            search allocates hundreds of MB per trial at the slowest speed.

    Returns:
        (float, float, float): time of the plateau start, the fitted onset `shift`, and the
            normalised match cost in [0, 1]. All NaN if the trial is too short.
    """
    sel = (t >= -pre) & (t <= duration)
    t, rs = np.asarray(t)[sel], np.asarray(rs)[sel]
    if len(t) > max_samples:
        t, rs = (
            t[:: int(np.ceil(len(t) / max_samples))],
            rs[:: int(np.ceil(len(t) / max_samples))],
        )
    ramp = plateau_speed / acceleration
    max_shift = duration - ramp - min_plateau
    if len(t) < 10 or max_shift <= 0:
        return np.nan, np.nan, np.nan
    cap = max(cap_rel * plateau_speed, cap_abs)
    # `shift >= 0` is what makes t_plateau >= v/a true by construction
    shifts = np.arange(0.0, max_shift, shift_step)
    template = np.clip(
        acceleration * (t[None, :] - shifts[:, None]), 0.0, plateau_speed
    )
    cost = np.minimum(np.abs(rs[None, :] - template), cap).mean(axis=1) / cap
    best = int(np.argmin(cost))
    return float(shifts[best] + ramp), float(shifts[best]), float(cost[best])


def trial_signals(rec, itrial, pre=2.0, post=0.5, fs_vol=FS_VOL):
    """Volume-rate and 1 kHz running speed of one trial, relative to the command."""
    epoch = rec["epochs"].iloc[int(itrial)]
    t0 = epoch.command_on
    duration = epoch.command_off - t0
    grid = np.arange(-pre, duration + post, 1 / fs_vol)
    position = np.interp(grid + t0, rec["time"], rec["position"])
    sel = (rec["time"] >= t0 - pre) & (rec["time"] <= t0 + duration + post)
    return dict(
        duration=duration,
        t_vol=grid[:-1] + 0.5 / fs_vol,
        rs_vol=np.diff(position) / np.diff(grid),
        t_khz=rec["time"][sel] - t0,
        rs_khz=rec["rs"][sel],
    )

In [ ]:
# Match every trial of every recording, on the volume-rate signal the pipeline sees and on
# the 1 kHz trace as the best available reference for the same trial.
rows = []
for name in all_fits.recording.unique():
    rec_i = get_recording(name)
    for _, trial in rec_i["epochs"].iterrows():
        if not np.isfinite(trial.plateau_speed):
            continue
        v = trial.plateau_speed
        sig = trial_signals(rec_i, trial.trial)
        t_vol, shift_vol, cost_vol = matchramp_template(
            sig["t_vol"], sig["rs_vol"], v, sig["duration"]
        )
        t_khz, shift_khz, cost_khz = matchramp_template(
            sig["t_khz"], sig["rs_khz"], v, sig["duration"]
        )
        rows.append(
            dict(
                recording=name,
                trial=int(trial.trial),
                plateau_speed=v,
                command_duration=sig["duration"],
                t_plateau=t_vol,
                onset=shift_vol,
                cost=cost_vol,
                t_plateau_khz=t_khz,
                onset_khz=shift_khz,
            )
        )
plateau = pd.DataFrame(rows)
plateau["lower_bound"] = plateau.plateau_speed / A_FIXED
plateau["model_opens"] = MODEL_ACC_TIME * plateau.plateau_speed + MODEL_MARGIN
plateau["model_error"] = plateau.model_opens - plateau.t_plateau
plateau["analysed_duration"] = plateau.command_duration - plateau.t_plateau
plateau["agree_volumes"] = (plateau.t_plateau - plateau.t_plateau_khz).abs() * FS_VOL

# The physical bound is a property of the parameterisation now, not a clamp -- assert it
# per trial rather than inspecting group summaries.
finite = plateau.t_plateau.notna()
assert (plateau.loc[finite, "onset"] >= 0).all(), "negative onset shift"
assert (
    plateau.loc[finite, "t_plateau"] >= plateau.loc[finite, "lower_bound"] - 1e-9
).all(), "plateau start earlier than v/a"
print(
    f"{finite.sum()}/{len(plateau)} trials matched; "
    f"shift >= 0 and t_plateau >= v/a hold for all of them\n"
)

print(
    plateau.groupby("plateau_speed")
    .agg(
        n=("trial", "count"),
        detected=("t_plateau", lambda x: x.notna().mean()),
        median=("t_plateau", "median"),
        iqr=("t_plateau", lambda x: x.quantile(0.75) - x.quantile(0.25)),
        onset_med=("onset", "median"),
        cost_med=("cost", "median"),
        same_volume=("agree_volumes", lambda x: (x < 0.5).mean()),
        within_1_vol=("agree_volumes", lambda x: (x < 1.5).mean()),
        model=("model_opens", "first"),
        frac_model_too_early=("model_error", lambda x: (x < 0).mean()),
        analysed_s=("analysed_duration", "median"),
    )
    .round(3)
    .to_string()
)
print()
print(
    "late outliers are kept, not rejected: the genuine 2-5 s late starts at low speed "
    "are real"
)

`onset_med` is a check the detector never sees: the recovered onset latency should land
near the ~0.5 s that the trapezoid fit of section B measures independently, on the same
trials, by a completely different route. `same_volume` and `within_1_vol` compare the
volume-rate answer with the 1 kHz answer for the same trial — the frame-rate one is what
the pipeline will use, the 1 kHz one is the best reference available.


### Why the loss is capped

The cap is the one design choice that is not forced by the physics, so it is worth showing
what it buys. Refit every trial with a plain squared loss and compare, grouped by how much
of the motor-on window the mouse spends with the wheel far below the target.


In [ ]:
def match_squared(
    t,
    rs,
    plateau_speed,
    duration,
    acceleration=A_FIXED,
    shift_step=SHIFT_STEP,
    min_plateau=MIN_PLATEAU_S,
    pre=0.5,
    max_samples=3000,
):
    """The same search under a plain squared loss, for comparison only."""
    sel = (t >= -pre) & (t <= duration)
    t, rs = np.asarray(t)[sel], np.asarray(rs)[sel]
    if len(t) > max_samples:
        k = int(np.ceil(len(t) / max_samples))
        t, rs = t[::k], rs[::k]
    ramp = plateau_speed / acceleration
    max_shift = duration - ramp - min_plateau
    if len(t) < 10 or max_shift <= 0:
        return np.nan
    shifts = np.arange(0.0, max_shift, shift_step)
    template = np.clip(
        acceleration * (t[None, :] - shifts[:, None]), 0.0, plateau_speed
    )
    cost = ((rs[None, :] - template) ** 2).mean(axis=1)
    return float(shifts[int(np.argmin(cost))] + ramp)


rows = []
for name in all_fits.recording.unique():
    rec_i = get_recording(name)
    for _, trial in rec_i["epochs"].iterrows():
        if not np.isfinite(trial.plateau_speed):
            continue
        v = trial.plateau_speed
        sig = trial_signals(rec_i, trial.trial)
        inside = (sig["t_vol"] >= v / A_FIXED) & (sig["t_vol"] <= sig["duration"])
        rows.append(
            dict(
                recording=name,
                trial=int(trial.trial),
                plateau_speed=v,
                # fraction of the window where the wheel is below half the commanded speed
                blocked=(
                    float((sig["rs_vol"][inside] < 0.5 * v).mean())
                    if inside.sum()
                    else np.nan
                ),
                t_squared=match_squared(
                    sig["t_vol"], sig["rs_vol"], v, sig["duration"]
                ),
            )
        )
squared = pd.DataFrame(rows).merge(
    plateau[["recording", "trial", "t_plateau"]], on=["recording", "trial"], how="left"
)
squared["gap"] = (squared.t_squared - squared.t_plateau).abs()
squared["bin"] = pd.cut(
    squared.blocked,
    [-0.01, 0.05, 0.2, 0.5, 1.01],
    labels=["<5%", "5-20%", "20-50%", ">50%"],
)

print(
    "blocked = fraction of the motor-on window with RS below half the commanded speed"
)
print(
    squared.groupby("bin", observed=True)
    .agg(
        n=("trial", "count"),
        median_gap_s=("gap", "median"),
        p90_gap_s=("gap", lambda x: x.quantile(0.9)),
        max_gap_s=("gap", "max"),
    )
    .round(3)
    .to_string()
)
print(
    f"\nsquared loss lands >1 s from the capped answer on {(squared.gap > 1).sum()}"
    f"/{squared.gap.notna().sum()} trials, >2 s on {(squared.gap > 2).sum()}"
)

The capped loss barely moves the typical trial — it prevents the rare catastrophic one.
That is what robustness buys here: the median gap is a few tens of ms, but on the trials
where the mouse fights the belt for a fifth to a half of the window the squared loss can
land **seconds** away, because one long blocked episode outweighs the entire rest of the
trace.


### Does it work on frame data alone?

The pipeline has no harp trace at hand — it has the volume-rate `RS` already in `trials_df`.
Load all four sessions the standard way with the acceleration cut disabled, so the full
motor-on window is kept, and run the matcher on `RS_stim` alone. This is the exact signal
`process_imaging_df` would use, differenced across the true (jittering) volume timestamps
rather than resampled onto a regular grid.


In [ ]:
frame_trials = []
for session_name in ALL_SESSIONS:
    _, trials_df_session = treadmill.sync_all_recordings(
        session_name=session_name,
        flexilims_session=flm,
        project=PROJECT,
        photodiode_protocol=5,
        filter_datasets={"anatomical_only": 3, "annotated": True},
        recording_type="two_photon",
        cut_trial_end=None,
        trial_duration=None,
        acceleration_time=None,
    )
    fs_session = flz.get_datasets(
        origin_name=session_name,
        dataset_type="suite2p_rois",
        filter_datasets=dict(annotated=True),
        flexilims_session=flm,
        allow_multiple=False,
    ).extra_attributes["fs"]
    trials_df_session = trials_df_session.reset_index(drop=True)
    # trial index within its recording, to match the harp-derived table. Note
    # trials_df carries BOTH `recording` (full name, set by treadmill.sync_all_recordings)
    # and `recording_name` (short genealogy name, set by generate_trials_df) -- the full
    # name is the one that matches the harp-derived table.
    trials_df_session["trial"] = trials_df_session.groupby("recording").cumcount()
    for _, trial in trials_df_session.iterrows():
        v = np.nanmedian(trial.MotorSpeed_stim)
        if not np.isfinite(v) or v <= 0:
            continue
        frame_trials.append(
            dict(
                session=session_name,
                recording=trial.recording,
                trial=int(trial.trial),
                plateau_speed=v,
                fs=fs_session,
                rs=np.asarray(trial.RS_stim, dtype=float) * 100,  # m/s -> cm/s
            )
        )
print(f"{len(frame_trials)} trials of frame data from {len(ALL_SESSIONS)} sessions")

In [ ]:
rows = []
for entry in frame_trials:
    t = np.arange(len(entry["rs"])) / entry["fs"]
    duration = t[-1] if len(t) else np.nan
    t_plateau, shift, cost = matchramp_template(
        t, entry["rs"], entry["plateau_speed"], duration
    )
    rows.append(
        dict(
            session=entry["session"],
            recording=entry["recording"],
            trial=entry["trial"],
            plateau_speed=entry["plateau_speed"],
            fs=entry["fs"],
            duration=duration,
            t_frames=t_plateau,
            onset_frames=shift,
            cost_frames=cost,
        )
    )
frames = pd.DataFrame(rows)

# Compare with the 1 kHz answer for the same trial -- a reference that shares none of the
# frame data's timestamp jitter.
frames = frames.merge(
    plateau[["recording", "trial", "plateau_speed", "t_plateau_khz"]],
    on=["recording", "trial", "plateau_speed"],
    how="left",
)
frames["agree_volumes"] = (frames.t_frames - frames.t_plateau_khz).abs() * frames.fs
print(
    f"{len(frames)} trials, matched to the 1 kHz reference: "
    f"{frames.t_plateau_khz.notna().mean():.3f}"
)
print(f"detected on frame data: {frames.t_frames.notna().mean():.3f}\n")
print(
    frames.groupby("plateau_speed")
    .agg(
        n=("trial", "count"),
        frames_median=("t_frames", "median"),
        khz_median=("t_plateau_khz", "median"),
        iqr=("t_frames", lambda x: x.quantile(0.75) - x.quantile(0.25)),
        onset_med=("onset_frames", "median"),
        cost_med=("cost_frames", "median"),
        same_volume=("agree_volumes", lambda x: (x < 0.5).mean()),
        within_1_vol=("agree_volumes", lambda x: (x < 1.5).mean()),
        analysed_s=("duration", "median"),
    )
    .round(3)
    .to_string()
)

# Ramp still admitted? The reference-free check: does the RAW speed dip well below the
# commanded value within the first 0.5 s of the window the detector opened?
admitted = []
for entry, row in zip(frame_trials, frames.itertuples()):
    if not np.isfinite(row.t_frames):
        admitted.append(np.nan)
        continue
    t = np.arange(len(entry["rs"])) / entry["fs"]
    head = entry["rs"][(t >= row.t_frames) & (t < row.t_frames + 0.5)]
    v = entry["plateau_speed"]
    admitted.append(((v - head.min()) / v > 0.2) if len(head) else np.nan)
frames["ramp_admitted"] = admitted
print("\nramp admitted (raw speed dips >20% below target in the first 0.5 s):")
print(
    frames.groupby("plateau_speed")
    .ramp_admitted.agg(["count", "mean"])
    .round(3)
    .to_string()
)

### Single trials, with the fitted template

Each panel shows the volume-rate running speed the pipeline sees, the 1 kHz trace behind it,
**the fitted template** and its cap envelope, the physically unreachable region before `v/a`
(grey), the detected plateau start (green) and, for contrast, where the current
`0.13·v + 0.5` model opens its window (red). The green span is the data that would be
analysed. A bad match is visible directly as the template not lying on the data.


In [ ]:
def plot_trial_plateau(ax, row, pre=2.0, legend=False):
    """Draw one trial with its fitted template and detected plateau start."""
    rec = get_recording(row.recording)
    sig = trial_signals(rec, row.trial, pre=pre)
    v = row.plateau_speed
    cap = max(CAP_REL * v, CAP_ABS)

    ax.axvspan(-pre, row.lower_bound, color="grey", alpha=0.12, lw=0)
    if np.isfinite(row.t_plateau):
        ax.axvspan(row.t_plateau, sig["duration"], color="green", alpha=0.08, lw=0)
    # Frame rate underneath, 1 kHz as a thin black line on top of it: the volume-rate trace
    # is what the detector sees, the 1 kHz one is what actually happened.
    ax.plot(
        sig["t_vol"],
        sig["rs_vol"],
        ".-",
        color="grey",
        lw=0.5,
        ms=2.5,
        alpha=0.7,
        zorder=1,
        label="RS (frame rate)",
    )
    ax.plot(
        sig["t_khz"],
        sig["rs_khz"],
        color="k",
        lw=0.35,
        alpha=0.85,
        zorder=2,
        label="RS (1 kHz)",
    )
    # The fitted template, and the band inside which deviations are scored proportionally
    if np.isfinite(row.onset):
        grid = np.linspace(-pre, sig["duration"], 400)
        model = ramp_template(grid, row.onset, v)
        ax.fill_between(
            grid,
            model - cap,
            model + cap,
            color="dodgerblue",
            alpha=0.15,
            lw=0,
            zorder=0,
            label=f"cap +/-{cap:.1f} cm/s",
        )
        ax.plot(
            grid, model, color="dodgerblue", lw=1.2, zorder=3, label="fitted template"
        )
    ax.axvline(0, color="k", lw=0.5, ls=":")
    if np.isfinite(row.t_plateau):
        ax.axvline(row.t_plateau, color="green", lw=1.2, label="Plateau detected")
    ax.axvline(
        row.model_opens, color="darkred", lw=1, ls="--", label="Model opens window"
    )

    ax.set_xlim(-pre, sig["duration"])
    ax.set_ylim(-0.2 * v, 1.9 * v)
    ax.text(
        0.02,
        0.97,
        f"{row.recording.split('_')[0]} #{int(row.trial)}  {v:.0f} cm/s",
        transform=ax.transAxes,
        va="top",
        fontsize=5.5,
    )
    ax.text(
        0.02,
        0.87,
        f"plateau {row.t_plateau:.2f} s   model {row.model_opens:.2f} s   "
        f"cost {row.cost:.2f}",
        transform=ax.transAxes,
        va="top",
        fontsize=5.5,
        color="darkred" if row.model_error < 0 else "k",
    )
    ax.tick_params(labelsize=5)
    ax.spines[["top", "right"]].set_visible(False)
    if legend:
        ax.legend(fontsize=4.5, frameon=False, loc="lower right")


def plot_plateau_grid(rows, ncols=5, title=None, filename=None, panel=(5.0, 3.6)):
    """Grid of single-trial panels, one per row of `rows`."""
    nrows = int(np.ceil(len(rows) / ncols))
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(ncols * panel[0] * cm, nrows * panel[1] * cm)
    )
    axes = np.atleast_1d(axes).ravel()
    for ax, (_, row) in zip(axes, rows.iterrows()):
        plot_trial_plateau(ax, row, legend=(ax is axes[0]))
    for ax in axes[len(rows) :]:
        ax.set_visible(False)
    if title:
        fig.suptitle(title, fontsize=9)
    fig.supxlabel("Time from motor command (s)", fontsize=6)
    fig.supylabel("Running speed (cm/s)", fontsize=6)
    plt.tight_layout()
    if filename:
        fig.savefig(save_folder / filename, bbox_inches="tight", transparent=True)
    return fig


detected = plateau[plateau.t_plateau.notna()].copy()
gallery = (
    detected.groupby("plateau_speed", group_keys=False)
    .apply(lambda g: g.sample(min(10, len(g)), random_state=0), include_groups=False)
    .assign(plateau_speed=lambda df: df.index.map(detected.plateau_speed))
    .sort_values(["plateau_speed", "trial"])
)
plot_plateau_grid(
    gallery,
    title="Detected plateau, 10 random trials per speed",
    filename="plateau_gallery.pdf",
)

**Worst matches.** The highest-`cost` trials, i.e. where the running speed least resembles
the expected profile. These should be the trials where the mouse never followed the belt —
the check that `cost` means what it claims to.


In [ ]:
worst = detected.nlargest(12, "cost")
plot_plateau_grid(
    worst,
    ncols=4,
    title="12 worst template matches (highest cost)",
    filename="plateau_worst_cost.pdf",
)
print(
    worst[
        [
            "recording",
            "trial",
            "plateau_speed",
            "t_plateau",
            "onset",
            "cost",
            "analysed_duration",
        ]
    ]
    .round(2)
    .to_string(index=False)
)

**Shortest analysed windows.** A late plateau leaves less data; at 61 cm/s the window
is short to begin with, so this is where a minimum-duration guard would bite.


In [ ]:
plot_plateau_grid(
    detected.nsmallest(12, "analysed_duration"),
    ncols=4,
    title="12 shortest resulting analysed windows",
    filename="plateau_shortest_window.pdf",
)
print(
    detected.groupby("plateau_speed")
    .analysed_duration.describe()[["count", "min", "25%", "50%", "max"]]
    .round(2)
    .to_string()
)

In [ ]:
1 + 1

In [ ]:
speeds = sorted(detected.plateau_speed.unique())

longest_per_speed = pd.concat(
    [
        detected[detected.plateau_speed == v].nlargest(5, "analysed_duration")
        for v in speeds
    ]
)
_ = plot_plateau_grid(
    longest_per_speed,
    ncols=5,
    title="5 longest analysed windows per plateau speed",
    filename="plateau_longest_window_per_speed.pdf",
)

In [ ]:
shortest_per_speed = pd.concat(
    [
        detected[detected.plateau_speed == v].nsmallest(5, "analysed_duration")
        for v in speeds
    ]
)
_ = plot_plateau_grid(
    shortest_per_speed,
    ncols=5,
    title="5 shortest analysed windows per plateau speed",
    filename="plateau_shortest_window_per_speed.pdf",
)

In [ ]:
gap = detected.t_plateau - detected.model_opens
mild_late = detected[(gap >= 0.2) & (gap <= 1.0)].sample(
    min(10, ((gap >= 0.2) & (gap <= 1.0)).sum()), random_state=0
)
plot_plateau_grid(
    mild_late,
    ncols=5,
    title="10 trials where the detected plateau is 0.2-1 s after the model opens",
    filename="plateau_mildly_late.pdf",
)
print(
    mild_late[
        [
            "recording",
            "trial",
            "plateau_speed",
            "t_plateau",
            "model_opens",
            "model_error",
        ]
    ]
    .round(2)
    .to_string(index=False)
)

**Detector.** Slide `clip(a·(t−s), 0, v)` over the trial, score with a mean absolute
deviation capped at `max(0.10·v, 1.5 cm/s)`, take the best `s ≥ 0`, and open the window at
`s + v/a`. One free parameter per trial; the `v/a` bound holds by construction; no tolerance,
hold time, smoothing window or rise limit. Measured on all four sessions (791 trials on the
harp traces, 788 on real `RS_stim` frame data).

|                                          | 3.8   | 7.6   | 15.25 | 30.5  | 61 cm/s |
| ---------------------------------------- | ----- | ----- | ----- | ----- | ------- |
| median plateau (s)                       | 1.06  | 1.53  | 2.45  | 4.44  | 8.37    |
| within 1 volume of 1 kHz, harp           | 1.000 | 0.981 | 0.994 | 1.000 | 1.000   |
| within 1 volume of 1 kHz, **frame data** | 0.961 | 0.975 | 0.975 | 0.994 | 0.981   |
| fitted onset latency (s)                 | 0.57  | 0.55  | 0.48  | 0.50  | 0.50    |
| median match cost                        | 0.28  | 0.30  | 0.30  | 0.23  | 0.16    |
| median analysed window (s)               | 9.43  | 8.97  | 8.03  | 6.02  | 2.11    |
| shortest analysed window (s)             | 1.01  | 3.91  | 6.26  | 5.30  | 1.39    |

Detection succeeds on **100%** of trials from `RS_stim`, the commanded speed and the volume
rate alone — no harp reload. The heuristic detector this replaces reached 0.62–0.91 within
one volume on the same harp comparison, and 0.71–0.96 on frame data.

**The fitted onset is independent corroboration, not a consistency check.** The matcher
treats the latency as a free parameter and is never told its value, yet it recovers
0.48–0.57 s at every commanded speed — the same ~0.5 s that section B's trapezoid fit
measures on the same trials by a completely different route.

**The cap is what makes it work on real trials.** Under a plain squared loss a single long
blocked-wheel episode outweighs the rest of the trace. Binned by the fraction of the window
with `RS` below half the commanded speed:

| blocked | n   | median gap | p90 gap    | max gap    |
| ------- | --- | ---------- | ---------- | ---------- |
| <5%     | 665 | 0.01 s     | 0.13 s     | 0.81 s     |
| 5–20%   | 90  | 0.01 s     | 0.38 s     | 3.02 s     |
| 20–50%  | 30  | 0.05 s     | **2.31 s** | **6.77 s** |
| >50%    | 6   | 0.39 s     | **4.02 s** | 4.61 s     |

So the cap does not change the typical trial — it prevents the rare catastrophic one:
10 of 791 trials land more than 2 s from the capped answer under squared error. Capping the
per-sample cost states that off-by-50% and off-by-100% are both simply bad, which is the
right statement about a mouse resisting the belt. Two variants were rejected on the data: a
pure 0/1 inlier count flattens the cost surface and loses the argmin (agreement falls to
0.82 overall, 0.59 at 61 cm/s), and a one-sided cap forgiving overshoot is degenerate, since
late shifts then cost nothing.

**Accepted trade-off.** `ramp_admitted` — the reference-free check that the raw speed does
not dip more than 20% below target in the first 0.5 s of the window — is 0.063, 0.056, 0.032
and 0.032 from 7.6 to 61 cm/s, at parity with the heuristic and better at 30.5. At 3.8 cm/s
it reads 0.413, but the metric is not meaningful there: its 20% threshold is 0.76 cm/s,
comparable to the plateau noise measured at that speed (~11% of the commanded speed), so it
flags the mouse pausing rather than the belt still climbing.

**Still outstanding for the pipeline.** The current model opens its window before the
plateau on **44–55% of trials**. And at 61 cm/s the analysed window left after the plateau is
2.11 s at the median and 1.39 s at worst, so a minimum-duration guard is needed — a property
of the experiment, not of this detection. The detector defines only where the window opens,
not its quality; trials that settle and then drop out when the mouse stops remain the job of
`max_abs_rs2motor_diff_ratio`, though `cost` now flags the worst of them.


## Verdict

**The acceleration is a hardware constant, and with it the start of the plateau follows from
a one-parameter fit to data the pipeline already holds.**

1. **The command is a step.** `MotorSps` reaches its target in one 7 ms frame-log sample, so
   the ramp is the controller's internal profile, not a commanded one.
2. **Acceleration is constant** at `a = 7.5–7.9 cm/s²` for every speed from 7.6 to 61 cm/s
   and for all four mice — between-mouse spread of the median 0.035 cm/s², 0.5% of the grand
   median, within-mouse IQR 0.11–0.27. `1/a = 0.128–0.130` s per cm/s, i.e. exactly the
   `acceleration_time = 0.13` already in the code. Nothing about it tracks the animal, so it
   is the motor, not behaviour.
3. **The current model's acceleration constant is right; its fixed margin is not.** With a
   0.5 s allowance the analysed window opens before the plateau on 44–55% of trials,
   at every commanded speed.
4. **The plateau start is recoverable per trial by template matching**, from `RS_stim`, the
   commanded speed and the volume rate alone. It fires on 100% of trials and agrees with the
   1 kHz answer to within one imaging volume on 96–99% of them.
5. **The fitted onset latency of 0.48–0.57 s** reproduces, as a free parameter, the ~0.5 s
   the trapezoid fit measures independently — so the match is locking onto the physical
   onset, not an artefact of the loss.
6. **A capped loss is required, not cosmetic.** It stops a blocked wheel from dominating the
   fit — under squared error 10/791 trials move by more than 2 s, up to 6.8 s — and it yields
   a per-trial match cost that flags the trials where the mouse never followed the belt.
7. **What still needs deciding at the pipeline level** is a minimum analysed duration: at
   61 cm/s only 2.11 s survives at the median, 1.39 s at worst.
